In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [0]:
df = spark.read.table('f1_racing.bronze.brz_results')

display(df.limit(4))

In [0]:
df.count()

In [0]:
df.filter(F.col("position") == '\\N').display()

In [0]:
df.withColumn('position',F.col('position')).display()

In [0]:
df = df.withColumn('number',F.when(F.col('number') == r'\N','-1')
                .otherwise(F.col('number')).cast(T.IntegerType()))\
    .withColumn('grid', F.col('grid').cast('int'))\
    .withColumn('position', F.when(F.col('position') == r'\N','-1')
                .otherwise(F.col('position')).cast('int'))\
     .withColumn('fastestLap', F.when(F.col('fastestLap') == r'\N','-1')
                .otherwise(F.col('fastestLap')).cast('int'))\
    .withColumn('rank',F.when(F.col('rank')==r'\N','-1')
                .otherwise(F.col('rank')).cast('int'))\
    .withColumn('milliseconds',F.when(F.col('milliseconds')==r'\N','-1')
                .otherwise(F.col('milliseconds')).cast('int'))\
    .withColumn('fastestLapSpeed',F.when(F.col('fastestLapSpeed')==r'\N','-1')
                .otherwise(F.col('fastestLapSpeed')).cast(T.DecimalType(10,3)))\
    .drop('source_fie','ingested_at')
    


In [0]:

df = df.filter(F.col('number') != -1)

In [0]:
df.count()

In [0]:
df.filter(F.col("position") == -1).display()

In [0]:
df.display()

In [0]:
display(df.limit(4))

In [0]:
df.select('time','fastestLapTime').show()

In [0]:

df = df.withColumn(
    "time",
    F.when(
        (F.col("milliseconds") > 0), 
        F.round(F.col("milliseconds") / 60000.0, 3)
    ).otherwise(None)
)

display(df)


In [0]:
lap_parts = F.split(F.col("fastestLapTime"), ":")

df = df.withColumn(
    "fastestLapTime",
    F.when(F.col("fastestLapTime") == r"\N", "-1.0")
    .otherwise(
        F.round(
            F.when(F.size(lap_parts) == 3,
                lap_parts.getItem(0).cast(T.DoubleType()) * 60.0 +
                lap_parts.getItem(1).cast(T.DoubleType()) +
                lap_parts.getItem(2).cast(T.DoubleType()) / 60.0
            ).when(F.size(lap_parts) == 2,
                lap_parts.getItem(0).cast(T.DoubleType()) +
                lap_parts.getItem(1).cast(T.DoubleType()) / 60.0
            ).otherwise(None),
            3
        ).cast(T.StringType())
    )
)


In [0]:
df = df.withColumn(
    'time',
    F.when(F.col('time').isNull(), -1.0).otherwise(F.col('time'))
)

df.show()

In [0]:
df.columns

In [0]:
new_names = {'resultId' : 'result_id',
 'raceId' : 'race_id',
 'driverId' : 'driver_id',
 'constructorId' : 'constructor_id',
 'positionText' : 'position_text',
 'positionOrder' : 'position_order',
 'fastestLap' : 'fastest_lap',
 'fastestLapTime' : 'fastest_lap_time',
 'fastestLapSpeed' : 'fastest_lap_speed',
 'statusId' : 'status_id'}

df = df.withColumnsRenamed(new_names)

display(df)

Databricks filtered table. Run in Databricks to view.

In [0]:
df.display()

In [0]:
# df = df.withColumn('number', F.when(F.col("number")== "\\N", -1).otherwise(F.col('number')))

In [0]:
df.columns

In [0]:
df = df.withColumn('position_text', F.when(F.col('position_text').isin(['D','E','F','N','R','W']),-1).otherwise(F.col('position_text')).cast('int'))

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
df = df.withColumn(
    "position_text", 
    F.coalesce(F.col("position_text").cast("int"), F.lit(-1))
)

In [0]:
df = df.withColumn('fastest_lap_time', F.col('fastest_lap_time').cast('float'))

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DecimalType

# Define your desired schema precisely
desired_schema = StructType([
    StructField("result_id", IntegerType(), True),
    StructField("race_id", IntegerType(), True),
    StructField("driver_id", IntegerType(), True),
    StructField("constructor_id", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("grid", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("position_text", IntegerType(), True), # If you want this as Integer now that it's cleaned
    StructField("position_order", IntegerType(), True),
    StructField("points", DoubleType(), True),
    StructField("laps", IntegerType(), True),
    StructField("time", DoubleType(), True),
    StructField("milliseconds", IntegerType(), True),
    StructField("fastest_lap", IntegerType(), True),
    StructField("rank", IntegerType(), True),
    StructField("fastest_lap_time", StringType(), True),
    StructField("fastest_lap_speed", DecimalType(10, 3), True),
    StructField("status_id", IntegerType(), True)
])

In [0]:
final_df = df

for field in desired_schema.fields:
    col_name = field.name
    col_type = field.dataType
    
    if col_name in final_df.columns:
        # Force cast the column to your desired type
        final_df = final_df.withColumn(col_name, final_df[col_name].cast(col_type))

# Verify the new schema matches perfectly
final_df.printSchema()

In [0]:
final_df.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('f1_racing.silver.slv_results')